# Avaliação Prática 2 — LSTM vs Transformer · executor Colab

Como na Avaliação 1: o notebook é o ambiente de execução, os scripts são o artefato.

**A base NÃO está no repositório** — é material da disciplina e não é redistribuída.
A célula 3 pede o upload de `g1_v1_ws.csv` e `g1_v2_ws.csv`.

**Antes de começar:** `Runtime → Change runtime type → T4 GPU`.
A varredura completa leva ~25 min.

In [ ]:
# 1. Clonar (ou atualizar) o repositório e instalar as dependências.
import pathlib, subprocess

REPO = pathlib.Path('/content/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-2'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)

assert ACTIVITY.is_dir(), f'{ACTIVITY} não existe no repositório publicado.'
%cd {ACTIVITY}
!pip install -q -r requirements.txt

In [ ]:
# 2. Confirmar a GPU. O BERT em CPU levaria horas; a LSTM e os baselines, não.
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
# 3. UPLOAD DA BASE — selecione g1_v1_ws.csv e g1_v2_ws.csv.
#    Os CSVs são material da disciplina: ficam fora do repositório público, então
#    entram por aqui, uma vez por sessão.
import shutil, pathlib
from google.colab import files

DATA = pathlib.Path('data'); DATA.mkdir(exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, DATA / name)
print(sorted(p.name for p in DATA.glob('*.csv')))

In [ ]:
# 4. Conferir o corpus antes de treinar qualquer coisa.
#    Aqui aparecem as 292 duplicatas e os 4 conflitos de rótulo — se este número mudar,
#    algo está errado com o upload, e é melhor descobrir agora que depois de 25 min de GPU.
import common

for task in ('binary', 'multiclass'):
    common.load_corpus(task)
    print()

In [ ]:
# 5. ETAPA CORE — as duas tarefas × {majoritária, TF-IDF+SVM, LSTM, BERT} (~8 min).
#    Ao fim desta célula a entrega já é completa: acurácia, F1 e matriz de confusão
#    para a tarefa binária e para a multiclasse, com LSTM e Transformer comparados.
!python run_all.py --stage core
!python report.py

In [ ]:
# 6. ETAPA SEEDS — sementes 7 e 2024 (~15 min).
#    O holdout 70/30 deixa ~730 textos de teste: o erro padrão binomial é ~1,5 pp.
#    Uma única partição pode ordenar dois modelos por sorte. Três sementes separam
#    diferença real de sorte.
!python run_all.py --stage seeds
!python report.py

In [ ]:
# 7. SENSIBILIDADE + EXTRAS (~6 min).
#    sensitivity: a tarefa binária SEM `neutro`/`surpresa` — testa se a conclusão
#                 depende da decisão do professor de chamá-los de 'positivo'.
#    extras:      BiLSTM — a bidirecionalidade resgata o braço recorrente?
!python run_all.py --stage sensitivity --stage extras
!python report.py

In [ ]:
# 8. Baixar os resultados para commitar (JSONs + matrizes de confusão).
!zip -q -r /content/ap2-results.zip results ../../assets/img/avaliacao-pratica-2-*.png

from google.colab import files
files.download('/content/ap2-results.zip')